# Initial dataset
The purpose of this notebook is to prepare an initial dataset for model training in the deployed application.  We start from the labeled headlines collected in notebooks 1 and 2, and proceed to clean and score them.

The training dataset is a dataframe of `sequence` (`str`), `embedding` (`np.array`), `relevant` (`0` or `1`), `score` (`0` or `1`), `user_feedback` (`bool`).

In [1]:
import pandas as pd

df = pd.read_csv("data/labeled_headlines.csv")
df = df[df.relevant != -1]
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)
df.reset_index(inplace=True)
df.drop(columns=["title", "description", "timestamp", "index"], inplace=True)

df

,concat,relevant
0,Are These the Bones of the Fourth Musketeer? T...,0
1,Trump Is in China as Iran War Continues With N...,1
2,A Tech Tycoon’s Prosecution Raises Fears of Au...,1
3,"Catherine, Princess of Wales, to Make First Of...",0
4,Why A.I. is the Hidden Minefield of Trump’s Ch...,1
...,...,...
2061,Youn Yuh-jung of “Beef” on the Tiffany Necklac...,0
2062,John Travolta Pulled Off a Beret. Could You? |...,0
2063,How Eric Kim Roasts a Chicken | Eric Kim brine...,0
2064,Pulled Pork at the Push of a Button | Assembli...,1


We add the embeddings

In [2]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("ProsusAI/finbert")
df["embedding"] = embedder.encode(df.concat.to_list(), show_progress_bar=True).tolist()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: ProsusAI/finbert
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/65 [00:00<?, ?it/s]

and we score the relevant headlines (`jittery` is set to 0 if the classifier is below 0.5, is set to 1 if above 0.8)

In [3]:
from transformers import pipeline
import torch

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else ("mps" if torch.mps.is_available() else "cpu")
)

print("Device:", device)

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device,
    dtype=torch.float16,
)

candidate_labels = ["jittery", "non-jittery"]

Device: mps


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [ ]:
from tqdm import tqdm

tqdm.pandas()


def scorer(row):
    if row["relevant"] != 1:
        return None

    result = classifier(row["concat"], candidate_labels=candidate_labels)
    score = dict(zip(result["labels"], result["scores"]))["jittery"]

    if score < 0.5:
        return 0
    if score > 0.8:
        return 1


df["jittery"] = df.progress_apply(scorer, axis=1)

100%|██████████| 2066/2066 [01:55<00:00, 17.85it/s] 


In [5]:
df = df.reindex(columns=["concat", "embedding", "relevant", "jittery"])

We add a column tracking whether the sentence is user feedback or not.  Later, user feedback will be used to retrain the model, and will be given heavier weight than automatically labeled entries.

In [6]:
df["user_feedback"] = False

The training dataset is saved as csv file.

In [7]:
df.to_csv("training.csv", index=False)

To use the dataset, load the csv file and evaluate the `embedding` column to convert it from string to list.

In [8]:
import ast

df = pd.read_csv("training.csv")
df["embedding"] = df.embedding.apply(ast.literal_eval)

df

,concat,embedding,relevant,jittery,user_feedback
0,Are These the Bones of the Fourth Musketeer? T...,"[0.09192849695682526, 0.2081747204065323, 0.06...",0,NaN,False
1,Trump Is in China as Iran War Continues With N...,"[-0.24815446138381958, -0.020585568621754646, ...",1,1.0,False
2,A Tech Tycoon’s Prosecution Raises Fears of Au...,"[-0.12966248393058777, -0.032048966735601425, ...",1,1.0,False
3,"Catherine, Princess of Wales, to Make First Of...","[-0.3431681990623474, 0.28847524523735046, 0.3...",0,NaN,False
4,Why A.I. is the Hidden Minefield of Trump’s Ch...,"[-0.11767198890447617, 0.24949780106544495, -0...",1,1.0,False
...,...,...,...,...,...
2061,Youn Yuh-jung of “Beef” on the Tiffany Necklac...,"[0.06608801335096359, 0.0398399792611599, 0.21...",0,NaN,False
2062,John Travolta Pulled Off a Beret. Could You? |...,"[0.2984325885772705, 0.33157432079315186, -0.2...",0,NaN,False
2063,How Eric Kim Roasts a Chicken | Eric Kim brine...,"[0.13721396028995514, 0.5959745645523071, -0.6...",0,NaN,False
2064,Pulled Pork at the Push of a Button | Assembli...,"[-0.10915020853281021, 0.2190592885017395, -0....",1,0.0,False
